In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib import util
os.makedirs('compare_out_test', exist_ok=True)
# Data paths (ajusta si es necesario)
PDF_REAL = r'C:/Users/BISITE-NEL/Desktop/pruebas/samples_real_22.npy'
PDF_SYNTH = r'C:/Users/BISITE-NEL/Desktop/pruebas/samples_synthetic_22.npy'
MY_GEN = r'C:/Users/BISITE-NEL/Desktop/pruebas/samples_mygen_22.npy'
print('Paths set')

In [ ]:
# Cargar arrays y mostrar shapes
A = np.load(PDF_REAL)
B = np.load(PDF_SYNTH)
C = np.load(MY_GEN)
print('PDF real shape:', A.shape)
print('PDF synth shape:', B.shape)
print('My gen shape:', C.shape)

In [ ]:
# Helpers: promedio de derivaciones y re-muestreo simple
def avg_leads(X):
    X = np.asarray(X)
    if X.ndim == 3:
        return X.mean(axis=1)
    if X.ndim == 2:
        return X
    raise ValueError('Unsupported shape')

def resample_to(X, target_len):
    import numpy as _np
    n, L = X.shape
    if L == target_len:
        return X
    x_old = _np.linspace(0,1,L)
    x_new = _np.linspace(0,1,target_len)
    return _np.vstack([_np.interp(x_new, x_old, row) for row in X])

In [ ]:
# Importar compute_metrics desde script por ruta
spec = util.spec_from_file_location('compute_additional_metrics', os.path.abspath('scripts/compute_additional_metrics.py'))
cam = util.module_from_spec(spec)
spec.loader.exec_module(cam)
compute_metrics = cam.compute_metrics

# Preparar datos (promediar derivaciones y alinear longitudes)
A2 = avg_leads(A)
B2 = avg_leads(B)
C2 = avg_leads(C)
target_len = A2.shape[1]
if B2.shape[1] != target_len:
    B2 = resample_to(B2, target_len)
if C2.shape[1] != target_len:
    C2 = resample_to(C2, target_len)

# Calcular métricas para tres pares
pairs = [ ('pdf_real_vs_pdf_synth', A2, B2), ('pdf_real_vs_mygen', A2, C2), ('pdf_synth_vs_mygen', B2, C2) ]
rows = []
for name, X, Y in pairs:
    m = compute_metrics(X, Y)
    m['pair'] = name
    rows.append(m)
df = pd.DataFrame(rows).set_index('pair')
df.to_csv('compare_out_test/three_datasets_summary.csv')
df

In [ ]:
# Guardar tabla y mostrarla bonita
from IPython.display import display
display(df)

# Graficar ejemplo: sample 0 (lead average) de cada conjunto
plt.figure(figsize=(10,6))
t = np.arange(target_len)
plt.plot(t, A2[0], label='PDF real')
plt.plot(t, B2[0], label='PDF synth')
plt.plot(t, C2[0], label='My gen')
plt.legend()
plt.title('Comparación señales (sample 0, lead-averaged)')
plt.tight_layout()
plt.savefig('compare_out_test/three_signal_comparison.png', dpi=150)
plt.show()

**Notas:** el notebook promedia derivaciones para simplificar la comparación y remuestrea las señales al mismo largo del conjunto `samples_real_22.npy` (1000 pasos).
Si prefieres comparar por derivación, edita la celda `avg_leads` para seleccionar la derivación deseada o eliminar el promedio.

In [ ]:
# Mostrar PNG y tabla desde compare_out_three
from IPython.display import Image, display
import pandas as pd
pngp = 'compare_out_three/three_signal_comparison.png'
csvp = 'compare_out_three/three_datasets_summary.csv'
print('PNG exists:', os.path.exists(pngp))
if os.path.exists(pngp):
    display(Image(filename=pngp))
if os.path.exists(csvp):
    df3 = pd.read_csv(csvp)
    display(df3)

In [ ]:
# Mostrar contenido JSON de métricas (si quieres ver detalles por par)
import json
for fn in ['pdf_real_vs_pdf_synth_metrics.json','pdf_real_vs_mygen_metrics.json','pdf_synth_vs_mygen_metrics.json']:
    p = os.path.join('compare_out_three', fn)
    if os.path.exists(p):
        print('---', fn)
        with open(p,'r') as f:
            print(json.dumps(json.load(f), indent=2))